# Does Dice tell us where the tumour is? Self-P2IR on all P2I-LReg patients

Pre-registered study: see `PREREGISTRATION.md` (written into `benchmark/` below). This notebook runs
the authors' pretrained Self-P2IR on every synthetic test view, then scores H1-H3 and E1.

**Before running**
1. *Add Input*: the dataset `junezhou001/p2i-lreg`.
2. *Settings*: **Internet on** (needs a phone-verified Kaggle account); accelerator GPU (T4 or P100) if available, CPU also works.
3. Kaggle keeps `/kaggle/working` only from a **Save Version -> Save & Run All** run, and a run stops after 12 hours.
   So split the runs over versions with `RUN_NOW` below, and pass each finished version's output to the next one
   (*Add Input* -> *Notebook Output*). Predictions found there are copied in and not recomputed.

In [ ]:
# ---- What this version runs -------------------------------------------------------------------
RUNS = {                                   # name -> arguments, fixed in PREREGISTRATION.md
    "A":        "--condition A",
    "B_recipe": "--condition B",
    "B_x1.05":  "--condition B --factor 1.05",
    "B_x1.11":  "--condition B --factor 1.11",
    "B_x1.32":  "--condition B --factor 1.32",
}
RUN_NOW = ["A", "B_recipe"]               # e.g. version 1: A + B_recipe, version 2: the three fixed factors
SCORE = False                             # True in the last version, once all five runs exist

## 1. Environment

In [ ]:
!nvidia-smi -L || echo "no GPU: running on CPU"
!pip install -q open3d easydict nibabel
import sys, torch, numpy, open3d
print(sys.version.split()[0], "torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "numpy", numpy.__version__, "open3d", open3d.__version__)

## 2. Find the dataset and the patients

In [ ]:
import glob, os
# e.g. /kaggle/input/datasets/junezhou001/p2i-lreg/Liver_regis/01/model/reconstructed_mesh_world_m.obj
hits = sorted(set(os.path.dirname(os.path.dirname(os.path.dirname(p)))
                  for depth in range(1, 7)
                  for p in glob.glob("/kaggle/input/" + "*/" * depth + "model/reconstructed_mesh_world_m.obj")))
assert len(hits) == 1, f"expected one dataset root, found {hits}"
ROOT = hits[0]
PATIENTS = sorted(p for p in os.listdir(ROOT) if p.isdigit() and
                  os.path.exists(f"{ROOT}/{p}/test_syn.txt") and os.path.exists(f"{ROOT}/{p}/syn/camPose.yml"))
print("dataset root:", ROOT)
print(len(PATIENTS), "patients:", " ".join(PATIENTS))
assert PATIENTS[0] == "01", "neighbourhood calibration follows main.py, which starts from patient 01"
PATIENT_ARGS = " ".join(PATIENTS)

## 3. Self-P2IR at the pre-registered commit, and its pretrained weights

In [ ]:
COMMIT = "60d6c1b30d57148e6bc0c97fd1fab0f68cc0f079"
WEIGHTS_SHA256 = "e83a9110bb6a1b1bf7e49e7e0f3a3736b0b01ce9ec76626972a5406afaa963ba"
REPO = "/kaggle/working/Self-P2IR"
WEIGHTS = "/kaggle/working/model_best_loss.pth"
if not os.path.exists(REPO):
    !git clone -q https://github.com/junzastar/Self-P2IR.git {REPO}
!git -C {REPO} checkout -q {COMMIT} && git -C {REPO} rev-parse HEAD
if not os.path.exists(WEIGHTS):
    !curl -sL -o {WEIGHTS} "https://drive.usercontent.google.com/download?id=1xz1u4jtc-J3g8IpgbjHMBS9CVpCCK4nO&export=download&confirm=t"
import hashlib
digest = hashlib.sha256(open(WEIGHTS, "rb").read()).hexdigest()
assert digest == WEIGHTS_SHA256, f"weights checksum {digest} differs from the pre-registered one"
print("weights checksum ok")

## 4. Our scripts (embedded from the repo by make_notebook.py)

In [ ]:
os.makedirs('/kaggle/working/benchmark/tools', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/benchmark/pilot_dice_vs_tumour.py
"""
Pilot: does a high Dice guarantee that the tumour is in the right place?

On synthetic P2I-LReg views (true camera pose known) we plant spherical tumours inside
the CT liver, produce wrong pose predictions, and score every prediction two ways:
  - Dice between the projected CT outline and the liver label (what papers report)
  - tumour error: mm between where the prediction puts each tumour and where it really is
A one-sided surface-fit score (visible cloud -> CT surface, mm) is reported as well,
since that is the other score that can be computed without ground truth.

Wrong predictions come from five families, all sampled the same way:
  camera   rotation about the camera centre + translation  (error in camera orientation)
  visible  rotation about the centroid of the visible surface + translation
  organ    rotation about the centroid of the whole liver + translation
  scale    zoom about the camera centre only (Dice is unchanged by construction; reported, not used)
  icp      point-to-plane ICP of the visible cloud to the CT, started from an 'organ' error.
           These are outputs of a real algorithm, so they are the primary test.
Rotation angle ~ U(0, 15 deg) about a uniformly random axis; translation ~ U(0, 20 mm)
in a uniformly random direction; scale ~ log-U(0.8, 1.25). Views are drawn at random from
each patient's test split, and tumours uniformly from the whole liver volume (not from the
visible part). All ranges and the decision rule below were fixed before the first run.

Pre-registered decision rule (Dice >= 0.90, mean tumour error over all tumours):
  SUPPORTED      >= 10 % of predictions have tumour error > 10 mm
  NOT SUPPORTED  <  2 % have tumour error > 10 mm AND Spearman |rho(Dice, error)| >= 0.8
  INCONCLUSIVE   otherwise

Amendment after a 6-view smoke test, before the full run: on the clean synthetic cloud, ICP
converged to < 0.5 mm from every start, so the 'icp' family cannot produce realistic failures.
Added 'icp_scaled': the same ICP run on the cloud scaled about the camera centre by
s ~ log-U(1.01, 1.32), the 10-90 % range of the PCA scale-recipe error we measured on 238
synthetic views. The decision rule is applied to 'icp_scaled'; 'icp' is still reported.
Simplifications: rigid only (synthetic livers are not deformed), spheres with guessed radius.

Usage: python benchmark/pilot_dice_vs_tumour.py [--patients 01 06 10 11] [--views 25] ...
"""
import argparse
import csv
import os

import cv2
import numpy as np
import open3d as o3d
import yaml

DATA_DIR = "dataset/liver-registration"
H, W = 540, 960                      # synthetic render size
LIVER, TIP, SHAFT = 0, 153, 179      # synthetic label values (102 = background)

MAX_ROT_DEG = 15.0
MAX_TRANS_MM = 20.0
SCALE_RANGE = (0.8, 1.25)
SCALE_RECIPE_ERROR = (1.01, 1.32)    # 10-90 % range of the PCA scale-recipe error on synthetic views
TUMOUR_RADIUS_MM = (5.0, 25.0)       # 1-5 cm diameter; to be replaced by LiTS statistics
DICE_GOOD = 0.90
MARGIN_MM = 10.0
FAMILIES = ["camera", "visible", "organ", "scale", "icp", "icp_scaled"]
PRIMARY = "icp_scaled"


# ==============================================================================
# Loading
# ==============================================================================
def load_patient(pid: str):
    """CT mesh in mm, ray-casting scene, ICP target, K, poses, test views."""
    root = os.path.join(DATA_DIR, pid)
    mesh = o3d.io.read_triangle_mesh(os.path.join(root, "model", "reconstructed_mesh_world_m.obj"))
    mesh.scale(1000.0, center=[0, 0, 0])
    mesh.compute_vertex_normals()

    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(mesh))

    target = mesh.sample_points_uniformly(200000, use_triangle_normal=True).voxel_down_sample(2.0)

    with open(os.path.join(root, "syn", "CAM_K.yml")) as f:
        K = np.array(yaml.safe_load(f)["00000"][0]["cam_K"]).reshape(3, 3)
    v, u = np.mgrid[0:H, 0:W]
    pix_dirs = np.stack([(u + 0.5 - K[0, 2]) / K[0, 0], (v + 0.5 - K[1, 2]) / K[1, 1],
                         np.ones((H, W))], axis=-1).reshape(-1, 3)  # one ray per pixel centre
    with open(os.path.join(root, "syn", "camPose.yml")) as f:
        poses = yaml.safe_load(f)
    with open(os.path.join(root, "test_syn.txt")) as f:
        views = [int(line.strip().replace("ptsLiverCam", "")) for line in f if line.strip()]

    return dict(root=root, mesh=mesh, centroid=np.asarray(mesh.vertices).mean(axis=0), scene=scene,
                target=target, K=K, pix_dirs=pix_dirs, poses=poses, views=views)


def true_w2c(pose) -> np.ndarray:
    """4x4 world(mm) -> OpenCV camera(mm), from Blender camera-to-world (-Z forward)."""
    R_cv = np.array(pose["cam_R_c2w"]).reshape(3, 3) @ np.diag([1, -1, -1])
    t = np.array(pose["cam_t_c2w"]).reshape(3)
    T = np.eye(4)
    T[:3, :3] = R_cv.T
    T[:3, 3] = -R_cv.T @ t
    return T


def load_view(P, idx: int):
    label = cv2.imread(os.path.join(P["root"], "syn", "labels", f"lbl{idx:05d}.png"))[..., 0]
    pcd = o3d.io.read_point_cloud(os.path.join(P["root"], "syn", "liverPcds", f"ptsLiverCam{idx:05d}.ply"))
    pcd = pcd.voxel_down_sample(3.0)
    pts = np.asarray(pcd.points) * np.array([1.0, -1.0, -1.0])  # Blender camera -> OpenCV camera
    return label, pts


# ==============================================================================
# Geometry helpers
# ==============================================================================
def apply(T: np.ndarray, X: np.ndarray) -> np.ndarray:
    return X @ T[:3, :3].T + T[:3, 3]


def project(X: np.ndarray, K: np.ndarray) -> np.ndarray:
    z = np.where(X[:, 2] > 1e-6, X[:, 2], np.nan)
    return np.stack([K[0, 0] * X[:, 0] / z + K[0, 2], K[1, 1] * X[:, 1] / z + K[1, 2]], axis=1)


def render_mask(T_pred: np.ndarray, P) -> np.ndarray:
    """Liver outline seen with pose T_pred: cast one ray per pixel into the CT mesh."""
    inv = np.linalg.inv(T_pred)
    dirs = P["pix_dirs"] @ inv[:3, :3].T
    rays = np.concatenate([np.broadcast_to(inv[:3, 3], dirs.shape), dirs], axis=1).astype(np.float32)
    t_hit = P["scene"].cast_rays(o3d.core.Tensor(rays))["t_hit"].numpy()
    return np.isfinite(t_hit).reshape(H, W)


def dice(pred: np.ndarray, liver: np.ndarray, ignore: np.ndarray) -> float:
    """Dice over pixels not covered by a tool (the liver behind a tool is unknown)."""
    p, g = pred & ~ignore, liver & ~ignore
    return 2.0 * (p & g).sum() / max(p.sum() + g.sum(), 1)


def random_rotation(rng, max_deg: float) -> np.ndarray:
    axis = rng.normal(size=3)
    axis /= np.linalg.norm(axis)
    a = np.deg2rad(rng.uniform(0, max_deg))
    Kx = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(a) * Kx + (1 - np.cos(a)) * Kx @ Kx


def rigid_error(rng, pivot: np.ndarray) -> np.ndarray:
    """4x4 camera-frame error: rotate about the pivot, then translate."""
    R = random_rotation(rng, MAX_ROT_DEG)
    d = rng.normal(size=3)
    t = d / np.linalg.norm(d) * rng.uniform(0, MAX_TRANS_MM)
    E = np.eye(4)
    E[:3, :3] = R
    E[:3, 3] = pivot - R @ pivot + t
    return E


def rotation_deg(R: np.ndarray) -> float:
    return np.rad2deg(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1)))


def sample_tumours(P, rng, n: int):
    """Sphere centres uniform in the liver volume, each sphere fully inside the liver."""
    lo, hi = P["mesh"].get_min_bound(), P["mesh"].get_max_bound()
    centres, radii = [], []
    while len(centres) < n:
        c = rng.uniform(lo, hi, size=(4096, 3)).astype(np.float32)
        r = rng.uniform(*TUMOUR_RADIUS_MM, size=4096)
        inside = P["scene"].compute_occupancy(o3d.core.Tensor(c), nsamples=9).numpy() > 0.5
        dist = P["scene"].compute_distance(o3d.core.Tensor(c)).numpy()
        ok = inside & (dist >= r)
        centres += list(c[ok].astype(np.float64))
        radii += list(r[ok])
    return np.array(centres[:n]), np.array(radii[:n])


def spearman(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return float("nan")
    rx, ry = np.argsort(np.argsort(x)), np.argsort(np.argsort(y))
    return float(np.corrcoef(rx, ry)[0, 1])


# ==============================================================================
# Scoring one predicted pose
# ==============================================================================
def surface_dists(T_pred, P, vis_pts):
    """Distance (mm) of each visible point, taken back to the CT frame with T_pred, to the CT surface."""
    back = apply(np.linalg.inv(T_pred), vis_pts).astype(np.float32)
    return P["scene"].compute_distance(o3d.core.Tensor(back)).numpy()


def evaluate(T_pred, T_true, P, view, tumours):
    label, vis_pts, in_view = view
    liver, ignore = label == LIVER, (label == TIP) | (label == SHAFT)

    d = dice(render_mask(T_pred, P), liver, ignore)

    fit = float(surface_dists(T_pred, P, vis_pts).mean())

    c_true, c_pred = apply(T_true, tumours), apply(T_pred, tumours)
    err = np.linalg.norm(c_pred - c_true, axis=1)
    lateral = np.linalg.norm((c_pred - c_true)[:, :2], axis=1)  # across the view; depth = |dz|
    px = np.linalg.norm(project(c_pred, P["K"]) - project(c_true, P["K"]), axis=1)
    return d, fit, err, lateral, px


def run_icp(T_init, P, vis_pts):
    src = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(vis_pts))
    res = o3d.pipelines.registration.registration_icp(
        src, P["target"], 20.0, np.linalg.inv(T_init),
        o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=60))
    return np.linalg.inv(res.transformation)


# ==============================================================================
# Main
# ==============================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--patients", nargs="+", default=["01", "06", "10", "11"])
    ap.add_argument("--views", type=int, default=25, help="random test views per patient that pass the check")
    ap.add_argument("--perts", type=int, default=40, help="wrong predictions per family per view")
    ap.add_argument("--tumours", type=int, default=20, help="tumours per patient")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="benchmark/pilot_out")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    rng = np.random.default_rng(args.seed)

    rows, per_tumour, excluded = [], [], []
    for pid in args.patients:
        P = load_patient(pid)
        tumours, radii = sample_tumours(P, rng, args.tumours)
        n_used = 0

        for idx in rng.permutation(P["views"]):
            if n_used == args.views:
                break
            label, vis_pts = load_view(P, int(idx))
            T_true = true_w2c(P["poses"][f"{idx:05d}"][0])

            # Sanity check: at the true pose the outline and the cloud must match the CT.
            # Views 2000-2499 of patients 10 and 11 fail it: an extra object hides part of the
            # liver, is labelled background, and its surface is included in the liver cloud.
            d0 = dice(render_mask(T_true, P), label == LIVER, (label == TIP) | (label == SHAFT))
            far0 = (surface_dists(T_true, P, vis_pts) > 3.0).mean()
            if d0 < 0.98 or far0 > 0.02:
                excluded.append((pid, int(idx), d0, far0))
                continue
            n_used += 1

            # A tumour is "in view" if its true centre projects onto the true liver outline
            gt_mask = render_mask(T_true, P)
            uv = project(apply(T_true, tumours), P["K"])
            ok = np.isfinite(uv).all(axis=1)
            ui, vi_ = np.full(len(uv), -1), np.full(len(uv), -1)
            ui[ok], vi_[ok] = np.round(uv[ok, 0]).astype(int), np.round(uv[ok, 1]).astype(int)
            inb = ok & (ui >= 0) & (ui < W) & (vi_ >= 0) & (vi_ < H)
            in_view = np.zeros(len(uv), bool)
            in_view[inb] = gt_mask[vi_[inb], ui[inb]]
            view = (label, vis_pts, in_view)

            print(f"[{pid}] view {idx:05d} ({n_used}/{args.views})  true pose: Dice {d0:.3f}, "
                  f"{100 * far0:.1f}% far points, {in_view.sum()}/{len(tumours)} tumours in view")

            pivots = {"camera": np.zeros(3), "visible": vis_pts.mean(axis=0),
                      "organ": apply(T_true, P["centroid"][None])[0]}
            for fam in FAMILIES:
                for _ in range(args.perts):
                    if fam in pivots:
                        T_pred = rigid_error(rng, pivots[fam]) @ T_true
                    elif fam == "scale":
                        s = np.exp(rng.uniform(*np.log(SCALE_RANGE)))
                        T_pred = np.diag([s, s, s, 1.0]) @ T_true
                    elif fam == "icp":
                        T_init = rigid_error(rng, pivots["organ"]) @ T_true
                        T_pred = run_icp(T_init, P, vis_pts)
                    else:
                        # The method sees a cloud with the wrong scale and aligns it rigidly
                        s = np.exp(rng.uniform(*np.log(SCALE_RECIPE_ERROR)))
                        T_init = np.diag([s, s, s, 1.0]) @ rigid_error(rng, pivots["organ"]) @ T_true
                        T_pred = run_icp(T_init, P, vis_pts * s)

                    seen = (label, vis_pts * s, in_view) if fam == "icp_scaled" else view
                    d, fit, err, lateral, px = evaluate(T_pred, T_true, P, seen, tumours)
                    E = T_pred @ np.linalg.inv(T_true)
                    scale = float(np.cbrt(np.linalg.det(E[:3, :3])))
                    rows.append(dict(
                        patient=pid, view=int(idx), family=fam, dice=d, surface_fit_mm=fit,
                        rot_err_deg=rotation_deg(E[:3, :3] / scale), scale=scale,
                        liver_centroid_err_mm=float(np.linalg.norm(apply(E, pivots["organ"][None])[0] - pivots["organ"])),
                        tumour_err_mean_mm=float(err.mean()), tumour_err_max_mm=float(err.max()),
                        tumour_lateral_mean_mm=float(lateral.mean()),
                        tumour_err_inview_mm=float(err[in_view].mean()) if in_view.any() else float("nan"),
                        tumour_px_inview=float(np.nanmean(px[in_view])) if in_view.any() else float("nan"),
                        n_inview=int(in_view.sum())))
                    per_tumour.append(np.stack([err, radii, in_view], axis=1))

    with open(os.path.join(args.out, "pilot_results.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    np.save(os.path.join(args.out, "pilot_per_tumour.npy"), np.stack(per_tumour))
    with open(os.path.join(args.out, "excluded_views.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["patient", "view", "true_pose_dice", "far_point_fraction"])
        w.writerows(excluded)

    print(f"\n{len(excluded)} views drawn but excluded by the true-pose check "
          f"(see excluded_views.csv): " + ", ".join(
              f"{pid}: {sum(e[0] == pid for e in excluded)}" for pid in args.patients))
    summarize(rows, args.out)


def summarize(rows, out_dir):
    lines = []
    col = lambda key, sel: np.array([r[key] for r in sel], float)
    bins = [(0.0, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 0.95), (0.95, 1.01)]
    for fam in FAMILIES:
        sel = [r for r in rows if r["family"] == fam]
        dsc, err = col("dice", sel), col("tumour_err_mean_mm", sel)
        fit, inv = col("surface_fit_mm", sel), col("tumour_err_inview_mm", sel)
        lat = col("tumour_lateral_mean_mm", sel)
        lines.append(f"\n== {fam}  (n = {len(sel)}) ==")
        lines.append(f"Spearman rho  Dice vs tumour error: {spearman(dsc, err):+.2f}   "
                     f"surface fit vs tumour error: {spearman(fit, err):+.2f}")
        lines.append(f"{'Dice bin':<12}{'n':>6}{'median mm':>11}{'P90 mm':>9}{'> 10 mm':>9}"
                     f"{'in-view >10':>13}{'lateral >10':>13}")
        for lo, hi in bins:
            m = (dsc >= lo) & (dsc < hi)
            if not m.any():
                continue
            iv = inv[m][np.isfinite(inv[m])]
            lines.append(f"[{lo:.2f},{min(hi, 1):.2f}){m.sum():>6}{np.median(err[m]):>11.1f}"
                         f"{np.percentile(err[m], 90):>9.1f}{100 * (err[m] > MARGIN_MM).mean():>8.1f}%"
                         f"{(100 * (iv > MARGIN_MM).mean() if len(iv) else float('nan')):>12.1f}%"
                         f"{100 * (lat[m] > MARGIN_MM).mean():>12.1f}%")

    sel = [r for r in rows if r["family"] == PRIMARY]
    dsc, err = col("dice", sel), col("tumour_err_mean_mm", sel)
    good = dsc >= DICE_GOOD
    frac = (err[good] > MARGIN_MM).mean() if good.any() else float("nan")
    rho = spearman(dsc, err)
    if not good.any():
        verdict = f"INCONCLUSIVE (no {PRIMARY} prediction reached Dice >= {DICE_GOOD})"
    elif frac >= 0.10:
        verdict = "SUPPORTED"
    elif frac < 0.02 and abs(rho) >= 0.8:
        verdict = "NOT SUPPORTED"
    else:
        verdict = "INCONCLUSIVE"
    lines.append(f"\nPre-registered test ({PRIMARY}, Dice >= {DICE_GOOD}): n = {good.sum()}, "
                 f"{100 * frac:.1f}% with tumour error > {MARGIN_MM:.0f} mm, rho = {rho:+.2f} -> {verdict}")

    text = "\n".join(lines)
    print(text)
    with open(os.path.join(out_dir, "pilot_summary.txt"), "w") as f:
        f.write(text + "\n")
    plot(rows, out_dir)


def plot(rows, out_dir):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib not installed; skipping the figure (pip install matplotlib)")
        return

    ink, muted, mark = "#2b2b2b", "#8a8a85", "#2a78d6"
    fams = [f for f in FAMILIES if f != "scale"]
    fig, axes = plt.subplots(2, len(fams), figsize=(4 * len(fams), 7), sharey=True)
    for j, fam in enumerate(fams):
        sel = [r for r in rows if r["family"] == fam]
        err = np.array([r["tumour_err_mean_mm"] for r in sel])
        for i, (key, xlabel) in enumerate([("dice", "Dice"), ("surface_fit_mm", "Surface fit (mm)")]):
            ax = axes[i, j]
            ax.scatter([r[key] for r in sel], err, s=6, alpha=0.35, color=mark, linewidths=0)
            ax.axhline(MARGIN_MM, color=muted, lw=1, ls="--")
            if key == "dice":
                ax.axvline(DICE_GOOD, color=muted, lw=1, ls="--")
            ax.set_xlabel(xlabel, color=ink)
            if j == 0:
                ax.set_ylabel("Tumour error (mm)", color=ink)
            if i == 0:
                ax.set_title(fam, color=ink)
            for s in ("top", "right"):
                ax.spines[s].set_visible(False)
            ax.tick_params(colors=muted)
    fig.suptitle("Dice and surface fit against tumour error (dashed: 10 mm margin, Dice 0.90)", color=ink)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "pilot_scatter.png"), dpi=130)
    print(f"figure saved to {os.path.join(out_dir, 'pilot_scatter.png')}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/benchmark/scale_recipe.py
"""
The PCA scale recipe of Self-P2IR (Zhou et al., TMI 2025, Sec. III-C, Eq. 1-2), as written in the paper.

Monocular depth has no true size, so the camera cloud T is resized by
    sigma = size(S) / size(T)
where S is the CT model and size() projects the points onto their own principal axis (the
eigenvector of the covariance with the largest eigenvalue). The paper does not say how the
projections become one number; we use their range (max - min), fixed before any run.

Applied to a synthetic cloud (true size), the recipe returns the factor by which it would
resize that cloud: 1.0 means the recipe is exact for that view.

Validity check (python benchmark/scale_recipe.py): if the released real PLYs were made with
this recipe, size(CT) / size(PLY) would be exactly 1 on every frame.
"""
import os

import cv2
import numpy as np
import open3d as o3d

DATA_DIR = "dataset/liver-registration"


def principal_size(X: np.ndarray, how: str = "range") -> float:
    Xc = X - X.mean(axis=0)
    _, V = np.linalg.eigh(np.cov(Xc.T))
    proj = Xc @ V[:, -1]
    return float(np.ptp(proj) if how == "range" else proj.std())


def recipe_factor(ct_vertices: np.ndarray, cloud: np.ndarray, how: str = "range") -> float:
    """Factor the recipe applies to `cloud` (same units as `ct_vertices`)."""
    return principal_size(ct_vertices, how) / principal_size(cloud, how)


def main():
    for how in ("range", "std"):
        ratios = []
        for pid in sorted(os.listdir(DATA_DIR)):
            base = os.path.join(DATA_DIR, pid)
            if not os.path.isdir(os.path.join(base, "real", "liverPcds")):
                continue
            mesh = o3d.io.read_triangle_mesh(os.path.join(base, "model", "reconstructed_mesh_world_m.obj"))
            S = np.asarray(mesh.vertices) * 1000.0
            for fn in sorted(os.listdir(os.path.join(base, "real", "liverPcds"))):
                frame = fn[:-4]
                pts = np.asarray(o3d.io.read_point_cloud(os.path.join(base, "real", "liverPcds", fn)).points)
                mask = cv2.imread(os.path.join(base, "real", "labels", frame, "label.png")).sum(-1).reshape(-1) > 0
                ratios.append(recipe_factor(S, pts[mask], how))
        r = np.array(ratios)
        print(f"size(CT) / size(released PLY), {how:5s}: median {np.median(r):.3f}, "
              f"10-90 % {np.percentile(r, 10):.3f}-{np.percentile(r, 90):.3f}, "
              f"min {r.min():.3f}, max {r.max():.3f}  ({len(r)} frames; 1.000 everywhere = recipe reproduced)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/benchmark/selfp2ir_predict.py
"""
Run the authors' pretrained Self-P2IR rigid network on synthetic test views; save the predicted poses.

Follows their synthetic tester (tester.py, _synTester.test_thr) step by step:
  dataset item -> collate_fn_3dmatch -> Pipeline (posenet) -> Matching.get_match(thr=0.15)
  -> MatchMotionLoss.ransac_regist_coarse (weighted SVD) -> R, t  (CT metres -> Blender camera metres)
Neighbourhood limits are calibrated as in main.py: on the augmented training set, whose first
items are patient 01's, with setup_seed(0).

Deviations from their code, all needed to run it here or to keep the test clean:
  - Only the rigid network (posenet.* weights) is built, so pytorch3d and the real-data modules
    are not needed. Weights are loaded with strict=True (their loader uses strict=False).
  - CUDA-only extensions imported by the loss module (Chamfer, EMD) and the plotting module are
    replaced by stubs; none of them is used at test time.
  - Their PoseDataset hard-codes patients 1-21; the list is replaced by --patients.
  - Their top-level folders are registered as packages explicitly, so installed packages with the
    same name (Hugging Face `datasets` on Kaggle) cannot shadow them.
  - A view their loader rejects (< 100 liver pixels or < 1000 correspondences to the TRUE pose)
    is skipped and recorded. Their test loop silently replaces it with a random other view.
  - Without a GPU, tensor.cuda() is a no-op so their solver runs on the CPU.
  - Open3D's random generator (used by their CT point sampling) is seeded, so runs repeat exactly.
Condition B resizes the target cloud after the rejection check, so A and B keep the same views.
Progress is saved every 25 views; rerunning the same command resumes, reusing the saved
neighbourhood limits (Kaggle sessions stop after 12 hours).

Usage:
  python benchmark/selfp2ir_predict.py --repo <Self-P2IR checkout> --weights model_best_loss.pth \
      --condition A [--patients 01 06 10 11] [--out benchmark/selfp2ir_out]
  --condition B             recipe scale (scale_recipe.py, range definition)
  --condition B --factor 1.11  fixed scale factor (sensitivity runs)
"""
import argparse
import hashlib
import json
import os
import sys
import time
import types

import numpy as np
import open3d as o3d
import torch
import yaml

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from scale_recipe import recipe_factor  # noqa: E402

PATIENT_LINE = "self.patients = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21]"


def stub(name, **attrs):
    mod = types.ModuleType(name)
    mod.__dict__.update(attrs)
    sys.modules[name] = mod


def import_repo(repo):
    """Make their packages importable without the training-only CUDA extensions."""
    sys.path.insert(0, repo)

    # Their top-level folders have no __init__.py, so an installed package of the same name
    # (e.g. Hugging Face `datasets` on Kaggle) would win; register the repo folders explicitly
    for name in ("configs", "cpp_wrappers", "datasets", "extensions", "lib", "models"):
        sys.modules.pop(name, None)
        pkg = types.ModuleType(name)
        pkg.__path__ = [os.path.join(repo, name)]
        sys.modules[name] = pkg

    class TrainingOnly(torch.nn.Module):  # lib/loss.py builds one at import time
        def forward(self, *args, **kwargs):
            raise RuntimeError("training-only CUDA extension, not available here")

    stub("extensions.chamfer_distance.chamfer_distance", ChamferDistance=TrainingOnly)
    stub("extensions.earth_movers_distance.emd", EarthMoverDistance=TrainingOnly)
    stub("lib.vis")  # mayavi / skimage plotting, only called in debug branches
    try:
        import torchvision.transforms  # noqa: F401
    except ImportError:
        stub("torchvision")
        stub("torchvision.transforms")
    try:
        import scipy.misc  # noqa: F401
    except ImportError:
        stub("scipy.misc")

    # Their dataset hard-codes patients 1-21; read the list from a module variable instead
    path = os.path.join(repo, "datasets", "dataset.py")
    src = open(path).read()
    assert src.count(PATIENT_LINE) == 1, "dataset.py changed: patient list not found"
    mod = types.ModuleType("datasets.dataset")
    mod.__file__ = path
    mod.PATIENTS = None
    sys.modules["datasets.dataset"] = mod
    exec(compile(src.replace(PATIENT_LINE, "self.patients = PATIENTS"), path, "exec"), mod.__dict__)
    return mod


def load_config(repo, data_root, device):
    from easydict import EasyDict as edict
    from configs.models import architectures

    yaml.add_constructor("!join", lambda loader, node: "_".join(str(i) for i in loader.construct_sequence(node)),
                         Loader=yaml.Loader)
    with open(os.path.join(repo, "configs", "test", "main_config.yaml")) as f:
        config = edict(yaml.load(f, Loader=yaml.Loader))
    config.dataset = "syn"
    config.data_root = os.path.abspath(data_root)
    config.device = device
    config.kpfcn_config.architecture = architectures["syn"]
    return config


def build_model(config, weights):
    from models.lepard.lepard import Pipeline

    state = torch.load(weights, map_location="cpu", weights_only=False)["state_dict"]
    posenet = {k[len("posenet."):]: v for k, v in state.items() if k.startswith("posenet.")}
    model = Pipeline(config)
    model.load_state_dict(posenet, strict=True)
    return model.to(config.device).eval()


def to_device(batch, device):
    """Same rule as their tester loop."""
    for k, v in batch.items():
        if isinstance(v, list):
            if v and not isinstance(v[0], (str, np.ndarray)):
                batch[k] = [item.to(device) for item in v]
        elif not isinstance(v, (dict, float, type(None), np.ndarray)):
            batch[k] = v.to(device)
    return batch


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data-root", default="dataset/liver-registration")
    ap.add_argument("--patients", nargs="+", default=["01", "06", "10", "11"])
    ap.add_argument("--condition", choices=["A", "B"], required=True)
    ap.add_argument("--factor", type=float, default=None, help="fixed scale for B (default: recipe)")
    ap.add_argument("--limit", type=int, default=None, help="first N test views only (smoke test)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="benchmark/selfp2ir_out")
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        torch.Tensor.cuda = lambda self, *a, **k: self  # their solver calls .cuda() directly

    ds_mod = import_repo(os.path.abspath(args.repo))
    from datasets.dataloader import calibrate_neighbors, collate_fn_3dmatch
    from lib.logger import setup_seed
    from lib.loss import MatchMotionLoss as MML
    from models.lepard.matching import Matching as CM

    config = load_config(args.repo, args.data_root, device)
    patients = sorted(int(p) for p in args.patients)
    o3d.utility.random.seed(args.seed)

    tag = "A" if args.condition == "A" else ("B_recipe" if args.factor is None else f"B_x{args.factor:.2f}")
    os.makedirs(args.out, exist_ok=True)
    out_path = os.path.join(args.out, f"predictions_{tag}.json")
    records, rejected, limits = [], [], None
    if os.path.exists(out_path):
        saved = json.load(open(out_path))
        if saved["meta"].get("complete"):
            print(f"{out_path} is complete; delete it to rerun")
            return
        records, rejected = saved["predictions"], [tuple(x) for x in saved["meta"]["rejected"]]
        limits = np.array(saved["meta"]["neighbourhood_limits"])
        print(f"resuming {out_path}: {len(records)} done, {len(rejected)} rejected")
    done = {(r["patient"], r["view"]) for r in records} | set(rejected)

    # Neighbourhood limits, as in main.py: augmented training set, patient 01's views first
    if limits is None:
        setup_seed(0)
        ds_mod.PATIENTS = [patients[0]]
        train_set = ds_mod.PoseDataset("train", config, data_augmentation=True)
        limits = calibrate_neighbors(train_set, config.kpfcn_config, collate_fn=collate_fn_3dmatch)
    print("neighbourhood limits:", limits.tolist())

    model = build_model(config, args.weights)
    ds_mod.PATIENTS = patients
    test_set = ds_mod.PoseDataset("test", config, data_augmentation=False)
    ct_vertices = {p: np.asarray(o3d.io.read_triangle_mesh(
        os.path.join(config.data_root, f"{p:02d}", "model", "reconstructed_mesh_world_m.obj")).vertices)
        for p in patients}

    meta = dict(
        condition=tag, scale_definition="range" if args.factor is None else "fixed",
        repo=os.path.abspath(args.repo), weights_sha256=sha256(args.weights),
        device=str(device), torch=torch.__version__, neighbourhood_limits=limits.tolist(),
        patients=patients, frames="R, t map CT points (metres) to the Blender camera (metres, -Z forward); "
                                  "R_true, t_true are the same from camPose.yml")

    def save(complete):
        with open(out_path, "w") as f:
            json.dump(dict(meta={**meta, "complete": complete, "n_views": len(records), "rejected": rejected},
                           predictions=records), f)

    n = len(test_set) if args.limit is None else min(args.limit, len(test_set))
    t0 = time.time()
    for i in range(n):
        patient, view = int(test_set.list_patient[i]), int(test_set.list_rank[i])
        if (patient, view) in done:
            continue
        np.random.seed(args.seed + i)  # their item sampling uses the global numpy RNG
        item = test_set.get_item(i)
        if item is None:
            rejected.append((patient, view))
            continue

        scale = 1.0
        if args.condition == "B":
            if args.factor is None:
                raw = np.asarray(o3d.io.read_point_cloud(test_set.list_liverPcd[i]).points) / 1000.0
                scale = recipe_factor(ct_vertices[patient], raw)
            else:
                scale = args.factor
            item["intra_liver_pcd_tgt"] = item["intra_liver_pcd_tgt"] * np.float32(scale)

        batch = to_device(collate_fn_3dmatch([item], config.kpfcn_config, neighborhood_limits=limits), device)
        with torch.no_grad():
            data = model(batch)
            match_pred, _, _ = CM.get_match(data["conf_matrix_pred"], thr=0.15, mutual=False)
            rot, trn, _, _ = MML.ransac_regist_coarse(data["s_pcd"], data["t_pcd"], data["src_mask"],
                                                     data["tgt_mask"], match_pred, data)

        records.append(dict(
            patient=patient, view=view, scale=scale, n_matches=int(len(match_pred)),
            R=np.asarray(rot[0].detach().cpu(), float).tolist(),
            t=np.asarray(trn[0].detach().cpu(), float).reshape(3).tolist(),
            R_true=np.asarray(item["pose_R"], float).tolist(),
            t_true=np.asarray(item["pose_t"], float).reshape(3).tolist()))
        if len(records) % 25 == 0:
            save(complete=False)
            print(f"[{tag}] {i + 1}/{n} views, {len(rejected)} rejected, {time.time() - t0:.0f} s")

    save(complete=True)
    print(f"saved {len(records)} predictions ({len(rejected)} rejected) to {out_path}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/benchmark/score_selfp2ir.py
"""
Score Self-P2IR predictions (from selfp2ir_predict.py) as pre-registered in PREREGISTRATION.md.

  python benchmark/score_selfp2ir.py --check   reproduction check only: the authors' registration
                                               recall on condition A, no Dice or TRE
  python benchmark/score_selfp2ir.py           H1, H2, H3 and E1 on every predictions_*.json found

Uses the pilot's code for Dice (tool pixels ignored), distance, the true-pose check and probe
tumours, so the numbers are computed exactly as in the pilot.
"""
import argparse
import csv
import glob
import json
import os

import numpy as np

import pilot_dice_vs_tumour as pilot
from pilot_dice_vs_tumour import (LIVER, SHAFT, TIP, apply, dice, load_patient, load_view, project,
                                  render_mask, sample_tumours, spearman, surface_dists, true_w2c)

N_PROBES = 200
MARGIN_MM = 10.0
DICE_ABS = 0.90
TOP_FRACTION = 0.25
N_BOOT = 2000
BLENDER_TO_CV = np.diag([1.0, -1.0, -1.0, 1.0])


def to_cv_mm(R, t) -> np.ndarray:
    """Their pose (CT metres -> Blender camera metres) as our 4x4 (CT mm -> OpenCV camera mm)."""
    T = np.eye(4)
    T[:3, :3] = np.asarray(R)
    T[:3, 3] = np.asarray(t) * 1000.0
    return BLENDER_TO_CV @ T


def load_predictions(pred_dir):
    out = {}
    for path in sorted(glob.glob(os.path.join(pred_dir, "predictions_*.json"))):
        d = json.load(open(path))
        if d["meta"].get("complete") is False:
            raise SystemExit(f"{path} is a partial run; finish it (rerun the same command) before scoring")
        out[d["meta"]["condition"]] = d
    assert "A" in out, f"no predictions_A.json in {pred_dir}"
    limits = {c: tuple(d["meta"]["neighbourhood_limits"]) for c, d in out.items()}
    assert len(set(limits.values())) == 1, f"conditions used different neighbourhood limits: {limits}"
    print("neighbourhood limits (all conditions):", next(iter(limits.values())))
    return out


# ==============================================================================
# Reproduction check (their metric only)
# ==============================================================================
def reproduction_check(preds):
    A = preds["A"]
    by_patient = {}
    for r in A["predictions"]:
        by_patient.setdefault(r["patient"], []).append(r)
    dists = []
    for pid, recs in by_patient.items():
        P = load_patient(f"{pid:02d}")
        verts = np.asarray(P["mesh"].vertices)
        for r in recs:
            T_pred, T_true = to_cv_mm(r["R"], r["t"]), to_cv_mm(r["R_true"], r["t_true"])
            dists.append(np.linalg.norm(apply(T_pred, verts) - apply(T_true, verts), axis=1).mean())
    dists = np.array(dists)
    print(f"Reproduction check, condition A: {len(dists)} views, {len(A['meta']['rejected'])} rejected by their loader")
    for rho in (2, 5, 10, 20, 40):
        print(f"  registration recall at rho = {rho:>2} mm: {100 * (dists < rho).mean():5.1f} %")
    print(f"  mean CT point error: median {np.median(dists):.2f} mm, 90th pct {np.percentile(dists, 90):.2f} mm")
    print("  paper (Fig. 7a): > 92 % at rho = 5 mm")


# ==============================================================================
# Per-view scoring
# ==============================================================================
def score_views(preds):
    """One row per (condition, view) that passes the true-pose check in every condition."""
    patients = sorted({r["patient"] for d in preds.values() for r in d["predictions"]})
    rows, excluded = [], []
    for pid in patients:
        P = load_patient(f"{pid:02d}")
        probes, _ = sample_tumours(P, np.random.default_rng([0, pid]), N_PROBES)
        views = sorted({r["view"] for r in preds["A"]["predictions"] if r["patient"] == pid})
        recs = {c: {r["view"]: r for r in d["predictions"] if r["patient"] == pid} for c, d in preds.items()}

        for view in views:
            label, vis_pts = load_view(P, view)
            T_true = true_w2c(P["poses"][f"{view:05d}"][0])
            ignore = (label == TIP) | (label == SHAFT)
            d0 = dice(render_mask(T_true, P), label == LIVER, ignore)
            far0 = (surface_dists(T_true, P, vis_pts) > 3.0).mean()
            if d0 < 0.98 or far0 > 0.02 or any(view not in recs[c] for c in preds):
                excluded.append((pid, view, d0, far0))
                continue

            # Their ground truth must be the same pose as ours
            T_their = to_cv_mm(recs["A"][view]["R_true"], recs["A"][view]["t_true"])
            assert np.abs(T_their - T_true).max() < 1e-2, f"pose convention mismatch at {pid}/{view}"

            gt_mask = render_mask(T_true, P)
            uv = project(apply(T_true, probes), P["K"])
            ok = np.isfinite(uv).all(axis=1)
            u, v = np.full(len(uv), -1), np.full(len(uv), -1)
            u[ok], v[ok] = np.round(uv[ok, 0]).astype(int), np.round(uv[ok, 1]).astype(int)
            inb = ok & (u >= 0) & (u < gt_mask.shape[1]) & (v >= 0) & (v < gt_mask.shape[0])
            in_view = np.zeros(len(probes), bool)
            in_view[inb] = gt_mask[v[inb], u[inb]]

            c_true = apply(T_true, probes)
            for cond in preds:
                r = recs[cond][view]
                T_pred = to_cv_mm(r["R"], r["t"])
                c_pred = apply(T_pred, probes)
                err = np.linalg.norm(c_pred - c_true, axis=1)
                rows.append(dict(
                    condition=cond, patient=pid, view=view, scale=r["scale"],
                    dice=dice(render_mask(T_pred, P), label == LIVER, ignore),
                    distance_mm=float(surface_dists(T_pred, P, vis_pts * r["scale"]).mean()),
                    view_tre_mm=float(err.mean()),
                    tre=err, depth=np.abs(c_pred[:, 2] - c_true[:, 2]),
                    px=np.linalg.norm(project(c_pred, P["K"]) - project(c_true, P["K"]), axis=1),
                    in_view=in_view))
            print(f"[{pid:02d}] view {view:05d} scored")
    return rows, excluded


# ==============================================================================
# Hypotheses
# ==============================================================================
def boot_ci(rows, stat):
    """95 % interval from resampling patients (the independent units)."""
    pids = sorted({r["patient"] for r in rows})
    by = {p: [r for r in rows if r["patient"] == p] for p in pids}
    rng = np.random.default_rng(0)
    vals = []
    for _ in range(N_BOOT):
        sample = [r for p in rng.choice(pids, len(pids)) for r in by[p]]
        s = stat(sample)
        if np.isfinite(s):
            vals.append(s)
    return (np.percentile(vals, 2.5), np.percentile(vals, 97.5)) if vals else (np.nan, np.nan)


def good_sets(rows):
    d = np.array([r["dice"] for r in rows])
    return {"Dice >= 0.90": d >= DICE_ABS,
            "top 25 % Dice": d >= np.quantile(d, 1 - TOP_FRACTION)}


def miss_rate(rows, margin=MARGIN_MM):
    """% of (view, probe) pairs with TRE above the margin."""
    if not rows:
        return np.nan
    return 100 * (np.concatenate([r["tre"] for r in rows]) > margin).mean()


def h1(rows, lines):
    rates = {}
    for name, m in good_sets(rows).items():
        good = [r for r, g in zip(rows, m) if g]
        rate = miss_rate(good)
        lo, hi = boot_ci(rows, lambda s, name=name: miss_rate(
            [r for r, g in zip(s, good_sets(s)[name]) if g]))
        per_view = 100 * np.mean([r["view_tre_mm"] > MARGIN_MM for r in good]) if good else np.nan
        curve = " ".join(f"{mg}:{miss_rate(good, mg):.0f}" for mg in range(0, 11, 2)) if good else "-"
        lines.append(f"  H1 [{name}] {len(good)} views: {rate:.1f} % of probes > 10 mm (95 % CI {lo:.1f}-{hi:.1f}); "
                     f"> 5 mm {miss_rate(good, 5):.1f} %; per-view > 10 mm {per_view:.1f} %")
        lines.append(f"     % of probes over margin (mm:%): {curve}")
        rates[name] = rate
    r = list(rates.values())
    if all(np.isfinite(r)) and all(x >= 10 for x in r):
        verdict = "a good Dice is NOT enough"
    elif all(np.isfinite(r)) and all(x < 2 for x in r):
        verdict = "a good Dice IS enough for this model"
    else:
        verdict = "INCONCLUSIVE"
    lines.append(f"  H1 verdict: {verdict}")


def h2(rows, lines):
    stat = lambda s: spearman(np.array([r["dice"] for r in s]), np.array([r["view_tre_mm"] for r in s]))
    rho = stat(rows)
    lo, hi = boot_ci(rows, stat)
    a = abs(rho)
    verdict = "good proxy" if a >= 0.8 else ("partial" if a >= 0.4 else "weak") if np.isfinite(a) else "n/a"
    lines.append(f"  H2 Spearman(Dice, view TRE) = {rho:+.2f} (95 % CI {lo:+.2f} to {hi:+.2f}) -> {verdict}")


def e1(rows, lines):
    tre = np.array([r["view_tre_mm"] for r in rows])
    lines.append(f"  E1 Spearman with view TRE: Dice {spearman(np.array([r['dice'] for r in rows]), tre):+.2f}, "
                 f"distance {spearman(np.array([r['distance_mm'] for r in rows]), tre):+.2f} (exploratory)")


def extras(rows, lines):
    """Also promised by the pre-registration: on-screen part, probes in view only, per patient. No verdicts."""
    cat = lambda key, sel: np.concatenate([r[key][r["in_view"]] if sel else r[key] for r in rows])
    good = [r for r in rows if r["dice"] >= DICE_ABS]
    in_good = np.concatenate([r["tre"][r["in_view"]] for r in good]) if good else np.array([])
    lines.append(f"  also: on-screen part median {np.nanmedian(cat('px', False)):.1f} px; "
                 f"probes in view {100 * np.mean(cat('in_view', False)):.0f} %, their median TRE "
                 f"{np.median(cat('tre', True)):.1f} mm; Dice >= 0.90 views, in-view probes > 10 mm: "
                 f"{100 * (in_good > MARGIN_MM).mean() if in_good.size else np.nan:.1f} %")
    for pid in sorted({r["patient"] for r in rows}):
        rp = [r for r in rows if r["patient"] == pid]
        gp = [r for r in rp if r["dice"] >= DICE_ABS]
        lines.append(f"     patient {pid:02d}: {len(rp)} views, median Dice {np.median([r['dice'] for r in rp]):.3f}, "
                     f"median view TRE {np.median([r['view_tre_mm'] for r in rp]):.1f} mm, "
                     f"Dice >= 0.90 probes > 10 mm {miss_rate(gp):.1f} % ({len(gp)} views)")


def h3(by_cond, lines):
    A = {(r["patient"], r["view"]): r for r in by_cond["A"]}
    for cond, rows in by_cond.items():
        if cond == "A":
            continue
        pairs = [(A[(r["patient"], r["view"])], r) for r in rows if (r["patient"], r["view"]) in A]
        dd = np.array([b["dice"] - a["dice"] for a, b in pairs])
        dt = np.array([b["view_tre_mm"] - a["view_tre_mm"] for a, b in pairs])
        confirmed = abs(np.median(dd)) < 0.02 and np.median(dt) > 5.0
        lines.append(f"  H3 [{cond}] {len(pairs)} paired views: median change in Dice {np.median(dd):+.3f} "
                     f"(median |change| {np.median(np.abs(dd)):.3f}), median change in view TRE {np.median(dt):+.1f} mm "
                     f"-> {'CONFIRMED (Dice is blind)' if confirmed else 'not confirmed'}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--pred-dir", default="benchmark/selfp2ir_out")
    ap.add_argument("--data-root", default=pilot.DATA_DIR)
    ap.add_argument("--check", action="store_true", help="reproduction check only")
    args = ap.parse_args()
    pilot.DATA_DIR = args.data_root  # read by load_patient
    preds = load_predictions(args.pred_dir)

    if args.check:
        reproduction_check(preds)
        return

    rows, excluded = score_views(preds)
    by_cond = {c: [r for r in rows if r["condition"] == c] for c in preds}

    lines = [f"Self-P2IR, pre-registered analysis ({len(by_cond['A'])} views per condition; "
             f"{len(excluded)} excluded by the true-pose check or missing in a condition; "
             f"loader-rejected: {len(preds['A']['meta']['rejected'])})"]
    for cond, rows_c in by_cond.items():
        tre = np.concatenate([r["tre"] for r in rows_c])
        lines.append(f"\n== condition {cond}: median Dice {np.median([r['dice'] for r in rows_c]):.3f}, "
                     f"median probe TRE {np.median(tre):.1f} mm (depth part "
                     f"{np.median(np.concatenate([r['depth'] for r in rows_c])):.1f} mm) ==")
        h1(rows_c, lines)
        h2(rows_c, lines)
        e1(rows_c, lines)
        extras(rows_c, lines)
    lines.append("\n== H3: does Dice notice a wrong scale? ==")
    h3(by_cond, lines)

    text = "\n".join(lines)
    print("\n" + text)
    with open(os.path.join(args.pred_dir, "summary.txt"), "w") as f:
        f.write(text + "\n")
    with open(os.path.join(args.pred_dir, "per_view.csv"), "w", newline="") as f:
        keys = ["condition", "patient", "view", "scale", "dice", "distance_mm", "view_tre_mm"]
        w = csv.DictWriter(f, fieldnames=keys + ["probes_over_10mm_pct", "median_depth_mm", "median_px",
                                                 "probes_in_view", "in_view_tre_mm"])
        w.writeheader()
        for r in rows:
            iv = r["in_view"]
            w.writerow({**{k: r[k] for k in keys}, "probes_over_10mm_pct": 100 * (r["tre"] > MARGIN_MM).mean(),
                        "median_depth_mm": np.median(r["depth"]), "median_px": np.nanmedian(r["px"]),
                        "probes_in_view": int(iv.sum()), "in_view_tre_mm": r["tre"][iv].mean() if iv.any() else ""})
    with open(os.path.join(args.pred_dir, "excluded_views.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["patient", "view", "true_pose_dice", "far_point_fraction"])
        w.writerows(excluded)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/benchmark/tools/build_cpp_wrappers.py
"""
Build Self-P2IR's two KPConv C++ helpers (grid subsampling, radius neighbours) for this machine.

Their setup.py needs numpy.distutils (removed from recent NumPy/setuptools), and the wrapper glue
uses NumPy 1 C-API spellings. We copy cpp_wrappers to a build folder, add a small compatibility
shim to the two wrapper.cpp files (no change to the neighbour or subsampling algorithms), build
there, and copy the .so files next to the original sources so `import cpp_wrappers...` works.

Usage: python build_cpp_wrappers.py <Self-P2IR checkout> <build folder>
"""
import glob
import os
import shutil
import sys

import numpy as np
from setuptools import Extension, setup

SHIM = """
// --- NumPy 1/2 compatibility shim (added for this build only) ---
#ifndef NPY_IN_ARRAY
#define NPY_IN_ARRAY NPY_ARRAY_IN_ARRAY
#endif
#define PyArray_NDIM(o) PyArray_NDIM((PyArrayObject*)(o))
#define PyArray_DIM(o, i) PyArray_DIM((PyArrayObject*)(o), i)
#define PyArray_DATA(o) PyArray_DATA((PyArrayObject*)(o))
// -----------------------------------------------------------------
"""

MODULES = {
    "cpp_subsampling": ("grid_subsampling", ["grid_subsampling/grid_subsampling.cpp", "wrapper.cpp"]),
    "cpp_neighbors": ("radius_neighbors", ["neighbors/neighbors.cpp", "wrapper.cpp"]),
}


def main():
    repo, build = os.path.abspath(sys.argv[1]), os.path.abspath(sys.argv[2])
    src = os.path.join(repo, "cpp_wrappers")
    dst = os.path.join(build, "cpp_wrappers")
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns("*.so"))

    for folder, (name, sources) in MODULES.items():
        base = os.path.join(dst, folder)
        wrapper = os.path.join(base, "wrapper.cpp")
        text = open(wrapper).read()
        anchor = "#include <numpy/arrayobject.h>"
        assert anchor in text, f"{wrapper}: numpy include not found"
        open(wrapper, "w").write(text.replace(anchor, anchor + SHIM, 1))

        os.chdir(base)
        setup(name=name, script_args=["build_ext", "--inplace"],
              ext_modules=[Extension(name, sources=["../cpp_utils/cloud/cloud.cpp"] + sources,
                                     include_dirs=[np.get_include()], extra_compile_args=["-std=c++11"])])
        for so in glob.glob(os.path.join(base, f"{name}*.so")):
            shutil.copy2(so, os.path.join(src, folder))
            print("built", os.path.join(src, folder, os.path.basename(so)))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/benchmark/PREREGISTRATION.md
# Pre-registration: does Dice tell us where the tumour is? (Self-P2IR)

**Written:** 2026-11-05, before any Self-P2IR prediction has been run or looked at.
**Pilot this builds on:** [pilot_dice_vs_tumour.py](pilot_dice_vs_tumour.py), with invented pose errors. This study uses predictions from a real published model instead.

Any change to this file is listed under *Amendments* with its reason and whether it was made before or after the first run. Analyses dropped by an amendment are still reported.

## Question

Papers on P2I-LReg report Dice (do the 2D outlines overlap?) and a surface distance. Surgeons care about TRE: how far the shown tumour is from the real one. On real frames TRE cannot be measured, so we test on synthetic views, where it can, whether Dice and distance tell us anything about TRE.

## Model

- **Self-P2IR** (Zhou et al., IEEE TMI 2025), official code (github.com/junzastar/Self-P2IR) and the authors' pretrained weights, unchanged.
- Recorded before the run:
  - repo commit `60d6c1b30d57148e6bc0c97fd1fab0f68cc0f079` (2025-12-04);
  - weights `model_best_loss.pth` (the only file released, 182 MB), sha256 `e83a9110bb6a1b1bf7e49e7e0f3a3736b0b01ce9ec76626972a5406afaa963ba`;
  - what the file is: the real-data model at epoch 19. It holds `posenet` (the rigid network) plus the real-data modules. The optimiser holds no state for the `posenet` tensors, so `posenet` was frozen during real-data training (as `main.py` does after loading the synthetic model). The `posenet` weights are therefore the synthetic-trained rigid network, which is what we test;
  - training views: not stored in the file. We assume the standard split (`train_syn.txt`), so `test_syn.txt` views are unseen. This assumption is stated with the results.
- Code used: [selfp2ir_predict.py](selfp2ir_predict.py) runs their synthetic tester's steps unchanged (matching, `get_match(thr=0.15)`, weighted SVD). Its header lists every deviation needed to run it.
- The model returns a pose, possibly with scale. It is trained with random scale ×0.8–1.25, so its rotation can contain a scale. Every prediction is used as a similarity transform: CT point X → s·R·X + t.

## Data

- **Views:** all synthetic test views (`test_syn.txt`) of every patient available on Kaggle that the tested model has not seen in training. Not a subset.
- **Exclusions, fixed in advance:**
  - views that fail the true-pose check (true-pose Dice < 0.98, or > 2 % of cloud points more than 3 mm from the CT). This removes views 2000–2499 of patients 10 and 11, where an extra object is labelled background but included in the liver cloud;
  - views the official loader rejects (fewer than 100 liver pixels or fewer than 1000 correspondences to the true pose). Their test loop silently replaces such a view with a random other view; we skip and record it instead.
  - The number excluded per patient is reported.
- **Probe tumours:**
  - 200 per patient, seed 0;
  - centres uniform through the whole liver volume (not only the visible part);
  - radius 5–25 mm, each sphere fully inside the liver.
  - Each probe answers "if this patient's tumour were here, how far off would it be?" They are not a model of how many tumours a patient has.
- **Condition A, clean:** the input exactly as the official test code prepares synthetic views (true size, `scale = 1.0`).
- **Condition B, recipe scale:**
  - the same views, but the cloud is resized by the PCA scale recipe the authors use for real frames, applied to the synthetic cloud;
  - our recipe implementation is fixed and committed before the run;
  - as a validity check we report how closely it reproduces the released `real/scale/*.txt` values;
  - if the agreement is poor, B results are labelled "approximate recipe";
  - sensitivity runs with fixed factors ×1.05, ×1.11 and ×1.32 are also reported.

## Measures

| Measure | Definition |
|---|---|
| **Dice** (per view) | Overlap of the CT outline rendered with the predicted pose and the liver label, ignoring tool pixels (label 153, 179). Same code as the pilot. |
| **Distance** (per view) | Mean distance (mm) from the visible cloud, taken to the CT frame with the predicted pose, to the CT surface. |
| **TRE** (per probe tumour) | Distance (mm) between the predicted and true position of that tumour centre. |
| **View TRE** (per view) | Mean TRE over the 200 probes. Used wherever one number per view is needed. |

**Good Dice**, two definitions, each applied within a condition:
- **absolute:** Dice ≥ 0.90;
- **relative:** the top 25 % of views by Dice.

Also reported:
- the depth and on-screen (px) parts of TRE;
- TRE for probes in view only;
- results per patient;
- 95 % confidence intervals from resampling patients (patients, not views, are the independent units).

## Hypotheses and what each result means

H1 and H2 are run separately on condition A and condition B.

**H1: Can a good Dice be trusted?**
Among good-Dice views, the % of (view, probe) pairs with TRE > 10 mm (the 1 cm margin), i.e. the chance that a patient's tumour misses the margin when Dice looks good.
- **≥ 10 %** under both good-Dice definitions: a good Dice is not enough.
- **< 2 %** under both: a good Dice is enough for this model.
- **Otherwise** (between, or the two definitions disagree): inconclusive.

Also reported:
- the same at 5 mm, and as a curve over margins from 0 to 10 mm;
- the per-view version: % of good-Dice views with view TRE > 10 mm.

**H2: Does a better Dice mean a lower TRE?**
Spearman correlation between Dice and view TRE across all views.
- **|ρ| ≥ 0.8:** Dice is a good proxy.
- **0.4–0.8:** partial.
- **< 0.4:** weak.

**H3: When the scale is wrong, does Dice notice?**
On the same views, compare B with A: the median change in Dice and the median change in view TRE.
- **Confirmed (Dice is blind):** Dice changes by < 0.02 while view TRE grows by > 5 mm.
- **Not confirmed:** otherwise.

**E1: Which is the better warning sign, Dice or distance?** (exploratory: reported, no verdict)
Spearman correlation of each with view TRE, in A and in B, side by side.

## Expected outcome and results that count against the idea

**Expected:** the paper reports near-perfect synthetic poses (rotation error ~0.19°, translation error ~1.2 mm, Table I). So in condition A we expect high Dice, small TRE and few H1 failures. That would mean the model is accurate on clean data, not that Dice is a good measure. Condition B is the informative test.

Results against the idea, reported as such:
- **H1 < 2 % and H2 ≥ 0.8 in condition B:** even with realistic scale, Dice is a good enough stand-in for this model. The argument for a new metric then rests on other methods or other realistic corruptions.
- **H3 not confirmed:** the scale blind spot from the pilot does not appear with a real model. It is dropped as a headline claim.

## Comparability with published numbers

- The paper's Dice of 78.89 % is measured on real frames, after deformation, with five-fold cross-validation.
- Our Dice is measured on synthetic frames, rigid, with tool pixels ignored.
- The two cannot be compared directly, and no claim of the form "their Dice of X hides Y mm" will be made from this study.

## Rules

- Thresholds are not changed after seeing results.
- All results are reported, including those against the idea.
- Limits stated with the results:
  - synthetic livers are rigid (no deformation);
  - probe tumours are spheres placed uniformly (LiTS location and size statistics come later);
  - the thresholds (10 %, 2 %, 0.8, 5 mm, 0.02, top 25 %) are our choices, made before the run, not literature standards. Only the 10 mm margin is clinical convention.

## Amendments

All made on 2026-11-05, **before any Self-P2IR run.**

1. **Probe tumours raised from 20 to 200 per patient.**
   A precision check on pilot predictions (240 predictions, patients 01 and 10, compared with 2000 probes) found that with 20 probes:
   - the view mean was off by 11.6 % in the worst 5 % of cases;
   - the "> 10 mm" verdict flipped for 2.4 % of predictions, about the size of the 2 % threshold in H1.
   Probes cost nothing to add.

2. **H1 counts each probe on its own instead of the view average.**
   The clinical question is the chance that the patient's tumour misses the margin, and an average can hide individual misses. The per-view version is still reported.

3. **H1 and H2 run on condition B as well as A.**
   The paper's near-perfect synthetic poses mean condition A alone would mostly show that the model is accurate on clean data.

4. **Relative good-Dice definition (top 25 %) added, required to agree with the absolute one.**
   If every view in a condition has Dice ≥ 0.90, the absolute line no longer separates good from bad.

5. **Released weights must be identified, and only unseen views used.**
   The paper trains real-data models per fold and uses patient-specific synthetic data, so the views available for testing depend on which models were released. (Done: see *Model*.)

Made on 2026-11-06, **before any Dice or TRE has been computed for Self-P2IR.**

6. **Condition B uses the recipe as written in the paper, with the range definition, labelled "approximate recipe".**
   The paper does not say how the projections onto the principal axis become one size. Validity check on all 84 local real frames ([scale_recipe.py](scale_recipe.py)): if the released clouds followed the recipe, size(CT) / size(PLY) would be exactly 1.000 on every frame. It is not:
   - range: median 1.112, 10–90 % 0.900–1.313;
   - std: median 1.379, 10–90 % 1.109–1.773.
   The released σ cannot be reproduced from the paper. Range is used because it is the closer of the two. As pre-registered, results from the fixed factors ×1.05, ×1.11 and ×1.32 are reported alongside.

7. **Reproduction check before the hypotheses.**
   Before any Dice or TRE is computed, condition A is compared with the paper on the authors' own registration recall: the % of views whose CT points land, on average, within ρ of their true position, with ρ = 2, 5, 10, 20, 40 mm. The paper reports > 92 % at ρ = 5 mm (Fig. 7a). If our run is far below that, the cause is found and fixed (or reported) before the hypotheses are tested.
   The paper's rotation error (Table I) is not used for this check: their code returns it in radians while the table labels it degrees, so the reference value is ambiguous.

8. **Rejected views are skipped, not replaced** (see *Data*). This differs from the authors' test loop, which substitutes a random view and so can count some views twice.

9. **The verdicts come only from the 21-patient run.**
   A first run on the 4 locally available patients (01, 06, 10, 11) was started on 2026-11-06 to test the whole pipeline end to end. Decided while it was running, before any Dice or TRE was computed:
   - its results are a dry run, reported as preliminary, and no verdict is declared from them;
   - the pre-registered verdicts come only from all patients on Kaggle, with the code and thresholds unchanged;
   - nothing in this file may be changed because of what the dry run shows.
   Reason: confidence intervals resample patients, and 4 patients give intervals too wide to judge the thresholds.

10. **Runs are seeded, including Open3D.**
    Their code samples CT points with Open3D's random generator, which it never seeds. The neighbourhood limits calibrated from those points changed between two runs, `[9, 21, 25, 29]` unseeded vs `[6, 15, 24, 29]` seeded, so their pipeline does not repeat exactly and the limits behind their published numbers cannot be known. We seed numpy, torch and Open3D (seed 0), calibrate as `main.py` does, and record the limits in every predictions file. All conditions of a run must share the same limits; the scorer reports them. The dry-run condition A, started before this change, is rerun with seeding.

Made on 2026-11-06, **after the Kaggle A and B_recipe predictions, before any Dice or TRE has been computed on the 21 patients.** A disclosure only; no analysis changes.

11. **Seeding makes runs repeat on one platform, not across platforms.**
    With the same seed, the neighbourhood limits are `[6, 15, 24, 29]` locally (CPU, macOS) and `[6, 16, 25, 29]` on Kaggle (GPU, Linux). On the 499 views both runs share (patients 01, 06, 10, 11) the predicted poses differ by a median of 0.19 mm and 0.17° (90th percentile 0.41 mm, 0.37°), and the same views are rejected. The confirmatory analysis uses only Kaggle predictions, so every condition shares the Kaggle limits (the scorer checks this).

## 5. Build the two C++ helpers

In [ ]:
%cd /kaggle/working
!python benchmark/tools/build_cpp_wrappers.py {REPO} /kaggle/working/build 2>&1 | grep -E "^built| error"

## 6. Predictions
Each run saves every 25 views; if a version stops, pass its output to the next version and the run resumes.

In [ ]:
import shutil
OUT = "/kaggle/working/selfp2ir_out"
os.makedirs(OUT, exist_ok=True)
previous = [p for depth in range(0, 6)                                      # outputs of earlier versions
            for p in glob.glob("/kaggle/input/" + "*/" * depth + "selfp2ir_out/predictions_*.json")]
for prev in previous:
    if not os.path.exists(os.path.join(OUT, os.path.basename(prev))):
        shutil.copy2(prev, OUT)
        print("copied", prev)
print(sorted(os.listdir(OUT)))

In [ ]:
import subprocess

def run(cmd, keep=()):
    """Run a command, show the lines containing `keep` (all if empty); on failure show the end and stop."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if not keep or any(k in line for k in keep):
            print(line, end="", flush=True)
    if p.wait() != 0:
        print("".join(tail))
        raise RuntimeError(f"command failed: {cmd}")

for name in RUN_NOW:
    print(f"===== {name} =====")
    run(f"python -W ignore benchmark/selfp2ir_predict.py --repo {REPO} --weights {WEIGHTS} --data-root {ROOT} "
        f"--patients {PATIENT_ARGS} {RUNS[name]} --out {OUT}",
        keep=("neighbourhood", "resuming", "views,", "saved", "complete"))

## 7. Reproduction check, then the pre-registered analysis
The check uses only the authors' own registration recall (paper: > 92 % at 5 mm). Look at it before scoring.

In [ ]:
if os.path.exists(os.path.join(OUT, "predictions_A.json")):
    run(f"python benchmark/score_selfp2ir.py --pred-dir {OUT} --data-root {ROOT} --check")
else:
    print("no condition A predictions yet: reproduction check skipped")

In [ ]:
if SCORE:
    run(f"python benchmark/score_selfp2ir.py --pred-dir {OUT} --data-root {ROOT} 2>&1 | grep -v 'scored$'")
else:
    print("SCORE is False: set it to True in the version that has all five runs")

## 8. Download

In [ ]:
!cd /kaggle/working && zip -qr selfp2ir_results.zip selfp2ir_out benchmark/PREREGISTRATION.md && ls -la selfp2ir_results.zip